In [1]:
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.io import fits
from astropy.time import Time

import scopesim as sim
import scopesim_templates as sim_tp
import importlib.resources
import pathlib
import warnings
warnings.filterwarnings('ignore')

try:
    irdb_path = str(pathlib.Path(importlib.resources.files('irdb')).parent)
except ModuleNotFoundError:
    irdb_path = str(pathlib.Path.home() / "src" / "irdb")

sim.rc.__config__["!SIM.file.local_packages_path"] = irdb_path
sim.rc.__config__["!SIM.file.search_path"].append(irdb_path)

sim.utils.set_console_log_level("INFO")

py.warnings - WARNING: /Users/yashvi/Desktop/ZShooter/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



#### Load simulator
Creating the optical train here and listing all effects to know what to tweak for different cases.

In [2]:
cmd = sim.UserCommands(use_instrument="ZShooter_v2", set_modes=["SPEC"])
zs = sim.OpticalTrain(cmd)
zs.effects

astar.scopesim.utils - Using !OBS.brightness dark for observation time
astar.scopesim.utils - WARNING: Obstime is not after sunset!
astar.scopesim.utils - Setting obstime to midnight of the same day. New UTC time is 2026-06-15T10:21:53.856.
astar.scopesim.utils - WARNING: Target coord !OBS.ra not set in !OBS config.
astar.scopesim.utils - WARNING: Target coord !OBS.dec not set in !OBS config.
astar.scopesim.utils - Setting coordinates from alt/airmass, location and obstime input.
astar.scopesim.effects.sky_ter_curves - WARNING: wmin 299.99999999999994 is below the minimum wavelength covered by SkyCalc. Setting to 300 nm.
astar.scopesim.utils - Using !OBS.brightness dark for observation time
astar.scopesim.utils - WARNING: Obstime is not after sunset!
astar.scopesim.utils - Setting obstime to midnight of the same day. New UTC time is 2026-06-15T10:21:53.856.
astar.scopesim.utils - WARNING: Target coord !OBS.ra not set in !OBS config.
astar.scopesim.utils - WARNING: Target coord !OBS.dec

element,name,class,included
str15,str31,str25,bool
MaunaKea,atmo_transmission,TERCurve,True
MaunaKea,continuum_emission,SkyBackgroundTERCurve,True
MaunaKea,airglow_and_interline_continuum,PalaceAirglowEmission,True
MaunaKea,seeing_psf,AOEnhanceablePSF,True
MaunaKea,adc_residuals,ADCShift,True
keck,telescope_reflection,SurfaceList,True
ZShooter,Selector,SurfaceList,True
ZShooter,dichroic_tree,DichroicTree,True
ZShooter,slitwheel_selector,SelectorWheel,True


#### Helper function for saving FITS with correct header info

In [3]:
def save_fits(hdul, extra_hdr, fileprefix):
    pathlib.Path("sims").mkdir(exist_ok=True)
    channels = ["B", "G", "R", "YJ", "H", "K"]
    for chan, hdu in zip(channels, hdul):
        hdr = hdu[1].header
        tag = 'blue' if chan=='B' else 'green' if chan=='G' else 'red'
        exptime = cmd[f'!OBS.dit_{tag}']*cmd[f'!OBS.ndit_{tag}']

        hdr.set("EXPTIME", exptime)
        hdr.set("DIT", cmd[f'!OBS.dit_{tag}'])
        hdr.set("NDIT", cmd[f'!OBS.ndit_{tag}'])

        mjd = Time(hdu[0].header['DATE'], format='isot').mjd
        hdr.set("MJD-OBS", mjd)
        hdr.update(extra_hdr)
        hdu[1].header = hdr
        hdu.writeto(f'sims/{fileprefix}_{chan}.fits', overwrite=True)

#### Dome flats

In [6]:
# create flat source
domeflat = sim_tp.calibration.flat_field(temperature=3000*u.K, amplitude=10*u.ABmag, filter_curve='V', extend=60)

# set exptimes for arms
cmd['!OBS.dit_blue'] = 600 # seconds
cmd['!OBS.dit_green'] = 30
cmd['!OBS.dit_red'] = 5

# Init optical train
zs = sim.OpticalTrain(cmd)

# turn off sky effects for dome flat
zs['atmo_transmission'].include = False
zs['continuum_emission'].include = False
zs['airglow_and_interline_continuum'].include = False
zs['seeing_psf'].include = False
zs['adc_residuals'].include = False

# observe
zs.observe(domeflat, update=True)

# readout
hdul = zs.readout()

# check desired counts and save
print(f'Max counts: {[np.max(hdu[1].data) for hdu in hdul]}')
extra_hdr = {"OBJECT": "DOMEFLAT", "IMAGETYPE": "LAMP,FLAT"}
save_fits(hdul, extra_hdr, "flat")

astar.scopesim.utils - Using !OBS.brightness dark for observation time
astar.scopesim.utils - WARNING: Obstime is not after sunset!
astar.scopesim.utils - Setting obstime to midnight of the same day. New UTC time is 2026-06-15T10:21:53.856.
astar.scopesim.utils - WARNING: Target coord !OBS.ra not set in !OBS config.
astar.scopesim.utils - WARNING: Target coord !OBS.dec not set in !OBS config.
astar.scopesim.utils - Setting coordinates from alt/airmass, location and obstime input.
astar.scopesim.effects.sky_ter_curves - WARNING: wmin 299.99999999999994 is below the minimum wavelength covered by SkyCalc. Setting to 300 nm.
astar.scopesim.utils - Using !OBS.brightness dark for observation time
astar.scopesim.utils - WARNING: Obstime is not after sunset!
astar.scopesim.utils - Setting obstime to midnight of the same day. New UTC time is 2026-06-15T10:21:53.856.
astar.scopesim.utils - WARNING: Target coord !OBS.ra not set in !OBS config.
astar.scopesim.utils - WARNING: Target coord !OBS.dec

 FOV effects: 100%|██████████| 3/3 [00:00<00:00, 48.64it/s]

astar.scopesim.optics.image_plane - No BUNIT found in added HDU.



 FOV effects: 100%|██████████| 3/3 [00:00<00:00, 27.53it/s]

astar.scopesim.optics.image_plane - No BUNIT found in added HDU.


...


 FOV effects: 100%|██████████| 3/3 [00:00<00:00, 28.01it/s]

astar.scopesim.optics.image_plane - No BUNIT found in added HDU.



 FOVs: 100%|██████████| 120/120 [00:21<00:00,  5.65it/s]

astar.scopesim.detector.detector_manager - Extracting from 1 detectors...


astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
Max counts: [np.float64(29140.68974701621), np.float64(35552.36546745118), np.float64(29947.801257096307), np.float64(49513.04943135876), np.float64(56190.24290895428), np.float64(53922.49746561219)]


#### Biases

In [8]:
cmd['!OBS.dit_blue'] = 0 # seconds
cmd['!OBS.dit_green'] = 0
cmd['!OBS.dit_red'] = 0

zs = sim.OpticalTrain(cmd)

# turn off sky effects (sim takes less time)
zs['atmo_transmission'].include = False
zs['continuum_emission'].include = False
zs['airglow_and_interline_continuum'].include = False
zs['seeing_psf'].include = False
zs['adc_residuals'].include = False

zs.observe(update=True)
hdul = zs.readout()

# check desired counts and save
print(f'Max counts: {[np.max(hdu[1].data) for hdu in hdul]}')
extra_hdr = {"OBJECT": "BIAS", "IMAGETYPE": "BIAS"}
save_fits(hdul, extra_hdr, "bias")

astar.scopesim.utils - Using !OBS.brightness dark for observation time
astar.scopesim.utils - WARNING: Obstime is not after sunset!
astar.scopesim.utils - Setting obstime to midnight of the same day. New UTC time is 2026-06-15T10:21:53.856.
astar.scopesim.utils - WARNING: Target coord !OBS.ra not set in !OBS config.
astar.scopesim.utils - WARNING: Target coord !OBS.dec not set in !OBS config.
astar.scopesim.utils - Setting coordinates from alt/airmass, location and obstime input.
astar.scopesim.effects.sky_ter_curves - WARNING: wmin 299.99999999999994 is below the minimum wavelength covered by SkyCalc. Setting to 300 nm.
astar.scopesim.utils - Using !OBS.brightness dark for observation time
astar.scopesim.utils - WARNING: Obstime is not after sunset!
astar.scopesim.utils - Setting obstime to midnight of the same day. New UTC time is 2026-06-15T10:21:53.856.
astar.scopesim.utils - WARNING: Target coord !OBS.ra not set in !OBS config.
astar.scopesim.utils - WARNING: Target coord !OBS.dec

 FOV effects: 100%|██████████| 3/3 [00:00<00:00, 51.20it/s]

astar.scopesim.optics.image_plane - No BUNIT found in added HDU.




 FOV effects: 100%|██████████| 3/3 [00:00<00:00, 29.40it/s]

astar.scopesim.optics.image_plane - No BUNIT found in added HDU.


...


 FOV effects: 100%|██████████| 3/3 [00:00<00:00, 28.25it/s]

astar.scopesim.optics.image_plane - No BUNIT found in added HDU.



 FOVs: 100%|██████████| 120/120 [00:20<00:00,  5.88it/s]

astar.scopesim.detector.detector_manager - Extracting from 1 detectors...


astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
Max counts: [np.float64(1050.1745608928964), np.float64(1050.1745608928964), np.float64(1050.1745608928964), np.float64(1050.1745608928964), np.float64(1050.1745608928964), np.float64(1050.1745608928964)]


#### Arcs

In [20]:
# create arc source
from astropy.table import Table

def create_arclamps(lines: Table, resol: float = 60000., scale_amp: float = 1e-5, wave_colname: str = 'wave', amp_colname: str = 'amplitude', wave_unit: str = 'Angstrom'):
    """
    Creates scopesim Source with spectrum of the input line list convolved to desired resolution.
    The lines have Gaussian profiles. The spatial field is uniformly illuminated.
    :param lines: astropy.table.Table of lines with columns for wavelength and amplitude
    :param resol: float, desired spectral resolution
    :param scale_amp: float, scaling factor for amplitude to get desired counts
    :param wave_colname: str, wavelength column name
    :param amp_colname: str, amplitude column name
    :param wave_unit: str, wavelength unit (by default Angstrom)
    :return:
    """
    import astropy.units as u
    from scopesim.source.source_templates import uniform_source, SourceSpectrum, Empirical1D
    import numpy as np

    if wave_colname+'_unit' in lines.meta:
        wave_unit = u.Unit(lines.meta[wave_colname+'_unit'])
    else:
        wave_unit = u.Unit(wave_unit)

    centers = np.array(lines[wave_colname]).astype(float)
    fwhms = 2.6*(centers/resol) # nyquist sampled
    stddevs = fwhms / (2*np.sqrt(2*np.log(2)))
    amps = np.array(lines[amp_colname]).astype(float) * scale_amp

    dw = (1*u.um).to_value(wave_unit) * 0.5/resol
    waves = np.arange(centers.min() - 2*dw, centers.max() + 2*dw, dw)
    # Fast sparse Gaussian accumulation — only touches pixels within ±10σ per line
    sp = np.zeros(len(waves))

    for cen, std, amp in zip(centers, stddevs, amps):
        hw = int(np.ceil(10 * std / dw))          # half-window in pixels
        idx = int(round((cen - waves[0]) / dw))  # center pixel
        lo = max(0, idx - hw)
        hi = min(len(waves), idx + hw + 1)
        w_local = waves[lo:hi]
        sp[lo:hi] += amp * np.exp(-(w_local - cen)**2 / (2 * std**2))

    return uniform_source(SourceSpectrum(Empirical1D, points=waves * wave_unit, lookup_table=sp), extent=60)

lines = Table.read('../notebooks/line_lists/combined_ubvisnir_lines.dat', delimiter='|', format='ascii', header_start=0)
arcs = create_arclamps(lines, 100000)

In [22]:
cmd['!OBS.dit_blue'] = 300
cmd['!OBS.dit_green'] = 120
cmd['!OBS.dit_red'] = 10

zs = sim.OpticalTrain(cmd)

# turn off sky effects for arcs
zs['atmo_transmission'].include = False
zs['continuum_emission'].include = False
zs['airglow_and_interline_continuum'].include = False
zs['seeing_psf'].include = False
zs['adc_residuals'].include = False

zs.observe(arcs, update=True)
hdul = zs.readout()

# check desired counts and save
print(f'Max counts: {[np.max(hdu[1].data) for hdu in hdul]}')
extra_hdr = {"OBJECT": "ARCS", "IMAGETYPE": "LAMP,ARC"}
save_fits(hdul, extra_hdr, "arcs")

astar.scopesim.utils - Using !OBS.brightness dark for observation time
astar.scopesim.utils - WARNING: Obstime is not after sunset!
astar.scopesim.utils - Setting obstime to midnight of the same day. New UTC time is 2026-06-15T10:21:53.856.
astar.scopesim.utils - WARNING: Target coord !OBS.ra not set in !OBS config.
astar.scopesim.utils - WARNING: Target coord !OBS.dec not set in !OBS config.
astar.scopesim.utils - Setting coordinates from alt/airmass, location and obstime input.
astar.scopesim.effects.sky_ter_curves - WARNING: wmin 299.99999999999994 is below the minimum wavelength covered by SkyCalc. Setting to 300 nm.
astar.scopesim.utils - Using !OBS.brightness dark for observation time
astar.scopesim.utils - WARNING: Obstime is not after sunset!
astar.scopesim.utils - Setting obstime to midnight of the same day. New UTC time is 2026-06-15T10:21:53.856.
astar.scopesim.utils - WARNING: Target coord !OBS.ra not set in !OBS config.
astar.scopesim.utils - WARNING: Target coord !OBS.dec

 FOV effects: 100%|██████████| 3/3 [00:00<00:00, 52.11it/s]

astar.scopesim.optics.image_plane - No BUNIT found in added HDU.




 FOV effects: 100%|██████████| 3/3 [00:00<00:00, 31.02it/s]

astar.scopesim.optics.image_plane - No BUNIT found in added HDU.


...


 FOV effects: 100%|██████████| 3/3 [00:00<00:00, 30.56it/s]

astar.scopesim.optics.image_plane - No BUNIT found in added HDU.



 FOVs: 100%|██████████| 120/120 [00:21<00:00,  5.64it/s]

astar.scopesim.detector.detector_manager - Extracting from 1 detectors...


astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
astar.scopesim.detector.detector_manager - Extracting from 1 detectors...
Max counts: [np.float64(17980.36953307334), np.float64(22827.75613095569), np.float64(32749.474998337115), np.float64(50168.39743666619), np.float64(14510.978531488277), np.float64(26318.668964610104)]
